In [ ]:
import pandas as pd

In [ ]:
!pip install transformers torch pandas tqdm sentencepiece -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_train = pd.read_parquet("/content/drive/MyDrive/[PROJECT] DATASET/df_train.parquet")
df_test  = pd.read_parquet("/content/drive/MyDrive/[PROJECT] DATASET/df_test.parquet")

print(f"Train shape: {df_train.shape}")
print(f"Test shape : {df_test.shape}")
print(f"\nTrain conversations: {df_train['conv_id'].nunique()}")
print(f"Test conversations : {df_test['conv_id'].nunique()}")

Train shape: (903607, 4)
Test shape : (2058781, 4)

Train conversations: 66927
Test conversations : 155128


In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model     = model.to("cuda" if torch.cuda.is_available() else "cpu")

print("Model loaded!")

# Fungsi translate manual
def translate_batch(texts, batch_size=32):
    results = []
    device  = next(model.parameters()).device

    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        batch = [t if t.strip() else '[KOSONG]' for t in batch]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        translated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("ind_Latn"),
            max_length=512
        )

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded!


In [ ]:
# Group per conversation
conv_texts = (
    df_train_to_translate
    .groupby('conv_id')['text']
    .apply(lambda msgs: ' [SEP] '.join(msgs.fillna('').tolist()))
    .reset_index()
)
conv_texts.columns = ['conv_id', 'combined_text']

print(f"Total conversations to translate: {len(conv_texts)}")

# Load checkpoint kalau ada
checkpoint_file = CHECKPOINT_DIR + "train_translate_checkpoint.json"
translated_dict = {}

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        translated_dict = json.load(f)
    print(f"Checkpoint loaded! {len(translated_dict)} conversations already translated")

# Filter yang belum ditranslate
remaining = conv_texts[~conv_texts['conv_id'].isin(translated_dict.keys())]
print(f"Remaining: {len(remaining)} conversations")

Total conversations to translate: 33464
Remaining: 33464 conversations


In [ ]:
# 1000 conversations
conv_labels = df_train.groupby('conv_id')['is_predator'].max().reset_index()

conv_sample = conv_labels.sample(n=1000, random_state=42)

# Split 500/500
conv_en, conv_id = train_test_split(
    conv_sample,
    test_size=0.5,
    random_state=42,
    stratify=conv_sample['is_predator']
)

df_train_en           = df_train[df_train['conv_id'].isin(conv_en['conv_id'])].reset_index(drop=True)
df_train_to_translate = df_train[df_train['conv_id'].isin(conv_id['conv_id'])].reset_index(drop=True)

print(f"English half : {df_train_en['conv_id'].nunique()} convs | {len(df_train_en)} messages")
print(f"To translate : {df_train_to_translate['conv_id'].nunique()} convs | {len(df_train_to_translate)} messages")

English half : 500 convs | 7465 messages
To translate : 500 convs | 7503 messages


In [ ]:
# Group per conversation
conv_texts = (
    df_train_to_translate
    .groupby('conv_id')['text']
    .apply(lambda msgs: ' [SEP] '.join(msgs.fillna('').tolist()))
    .reset_index()
)
conv_texts.columns = ['conv_id', 'combined_text']
print(f"Total conversations to translate: {len(conv_texts)}")

# Load checkpoint kalau ada
checkpoint_file = CHECKPOINT_DIR + "train_translate_checkpoint.json"
translated_dict = {}

if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        translated_dict = json.load(f)
    print(f"Checkpoint loaded! {len(translated_dict)} already done")

remaining = conv_texts[~conv_texts['conv_id'].isin(translated_dict.keys())]
print(f"Remaining: {len(remaining)} conversations")

# Translating
remaining_texts = remaining['combined_text'].tolist()
remaining_ids   = remaining['conv_id'].tolist()

for i in tqdm(range(0, len(remaining_texts), BATCH_SIZE), desc="Translating"):
    batch_ids   = remaining_ids[i:i+BATCH_SIZE]
    batch_texts = remaining_texts[i:i+BATCH_SIZE]
    batch_texts = [t if t.strip() else '[KOSONG]' for t in batch_texts]

    try:
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        translated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids("ind_Latn"),
            max_length=512,
            num_beams=1,
            early_stopping=True
        )

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)

        for conv_id, result in zip(batch_ids, decoded):
            translated_dict[conv_id] = result

    except Exception as e:
        print(f"Error batch {i}: {e}")
        for conv_id in batch_ids:
            translated_dict[conv_id] = ''

    # Checkpoint tiap 100 conversations
    if (i // BATCH_SIZE) % (100 // BATCH_SIZE) == 0 and i > 0:
        with open(checkpoint_file, 'w') as f:
            json.dump(translated_dict, f)
        print(f"\nCheckpoint saved! {len(translated_dict)} done")

# Final save checkpoint
with open(checkpoint_file, 'w') as f:
    json.dump(translated_dict, f)
print(f"Translation done! {len(translated_dict)} conversations")

# Map back
df_train_to_translate['text'] = df_train_to_translate['conv_id'].map(translated_dict)
df_train_id = df_train_to_translate.copy()

df_train_mix = pd.concat([df_train_en, df_train_id], ignore_index=True).sample(
    frac=1, random_state=42
).reset_index(drop=True)

df_train_en.to_parquet(SAVE_DIR  + "df_train_en.parquet")
df_train_id.to_parquet(SAVE_DIR  + "df_train_id.parquet")
df_train_mix.to_parquet(SAVE_DIR + "df_train_mix.parquet")

print("\n=== SAVED ===")
print(f"df_train_en  : {df_train_en.shape}")
print(f"df_train_id  : {df_train_id.shape}")
print(f"df_train_mix : {df_train_mix.shape}")

print("\n=== LABEL DISTRIBUTION ===")
print("EN  :\n", df_train_en['is_predator'].value_counts())
print("ID  :\n", df_train_id['is_predator'].value_comments())
print("MIX :\n", df_train_mix['is_predator'].value_counts())

Total conversations to translate: 500
Remaining: 500 conversations


Translating:  25%|██▌       | 4/16 [02:16<06:48, 34.06s/it]


Checkpoint saved! 128 done


Translating:  44%|████▍     | 7/16 [03:45<04:36, 30.76s/it]


Checkpoint saved! 224 done


...

Translating:  81%|████████▏ | 13/16 [06:42<01:29, 29.73s/it]


Checkpoint saved! 416 done


Translating: 100%|██████████| 16/16 [08:02<00:00, 30.17s/it]


Checkpoint saved! 500 done
Translation done! 500 conversations

=== SAVED ===
df_train_en  : (7465, 4)
df_train_id  : (7503, 4)
df_train_mix : (14968, 4)

=== LABEL DISTRIBUTION ===
EN  :
 is_predator
0    6949
1     516
Name: count, dtype: int64


AttributeError: 'Series' object has no attribute 'value_comments'

In [ ]:
print("ID  :\n", df_train_id['is_predator'].value_counts())
print("MIX :\n", df_train_mix['is_predator'].value_counts())

ID  :
 is_predator
0    7331
1     172
Name: count, dtype: int64
MIX :
 is_predator
0    14280
1      688
Name: count, dtype: int64


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Clear cache dulu
torch.cuda.empty_cache()